## 1. Vocabulary

In [1]:
vocab = {
    "<PAD>": 0,
    "<START>": 1,
    "I": 2,
    "love": 3,
    "AI": 4,
    "and": 5,
    "ML": 6,
    "<END>": 7
}

## 2. Import Tensorflow

In [2]:
import tensorflow as tf
from tensorflow.keras import layers

## 3. Model configuration

In [3]:
vocab_size = 8
embed_dim = 16
num_heads = 2
head_dim = embed_dim // num_heads
ff_dim = 32
max_sequence_length = 10

print("Vocabulary size:", vocab_size)
print("Embedding dimension:", embed_dim)
print("Number of heads:", num_heads)
print("Head dimension:", head_dim)
print("FFN dimension:", ff_dim)
print("Maximum sequence length:", max_sequence_length)

Vocabulary size: 8
Embedding dimension: 16
Number of heads: 2
Head dimension: 8
FFN dimension: 32
Maximum sequence length: 10


## 4. Token embedding

In [4]:
token_embedding = layers.Embedding(
    input_dim=vocab_size,
    output_dim=embed_dim
)

print("Token embedding created.")

Token embedding created.


## 5. Positional embedding

In [5]:
position_embedding = layers.Embedding(
    input_dim=max_sequence_length,
    output_dim=embed_dim
)

print("Position embedding created.")

Position embedding created.


## 6. Multi-Head Attention

In [6]:
attention_layer = layers.MultiHeadAttention(
    num_heads=num_heads,
    key_dim=head_dim
)

print("Attention layer created.")

Attention layer created.


## 7. Layer Normalization

In [7]:
layer_norm1 = layers.LayerNormalization()
layer_norm2 = layers.LayerNormalization()

print("Layer normalization layers created.")

Layer normalization layers created.


## 8. Feed Forward Network

In [8]:
ffn = tf.keras.Sequential([
    layers.Dense(ff_dim, activation="relu"),
    layers.Dense(embed_dim)
])

print("FFN created.")

FFN created.


## 9. Output Layer

In [9]:
output_layer = layers.Dense(vocab_size)

print("Output layer created.")

Output layer created.


## 10. Training data

In [10]:
tokens = tf.constant([[2, 3, 4, 5]])

print("Complete sequence:", tokens.numpy())

Complete sequence: [[2 3 4 5]]


## 11. Input vs Target shifting

In [11]:
inputs = tokens[:, :-1]
targets = tokens[:, 1:]

print("Inputs:", inputs.numpy())
print("Targets:", targets.numpy())

Inputs: [[2 3 4]]
Targets: [[3 4 5]]


## 12. Token embeddings

In [12]:
embedded = token_embedding(inputs)

print("Embedded shape:", embedded.shape)
print(embedded.numpy())

Embedded shape: (1, 3, 16)
[[[ 0.01411856  0.04757687  0.04224265 -0.03587254 -0.01149664
   -0.022872   -0.01345545 -0.0002871   0.03740505  0.02591484
   -0.03469913 -0.02721027 -0.03206496 -0.03367077 -0.03085026
   -0.02818301]
  [ 0.01244501 -0.00168001 -0.01249511  0.02918137 -0.04900923
    0.01059058 -0.04727049 -0.01267502 -0.02810135 -0.03062661
   -0.04191109  0.00428658  0.03288535 -0.02118421 -0.01696859
   -0.03761669]
  [ 0.01037157  0.04127919 -0.04374775  0.03361421  0.02424743
    0.03557995 -0.02384549 -0.03619244 -0.03621143 -0.02028728
   -0.0293273   0.02509749  0.00982666 -0.01218687 -0.04302137
   -0.03492542]]]


## 13. Positional embeddings

In [13]:
sequence_length = tf.shape(inputs)[1]

positions = tf.range(
    start=0,
    limit=sequence_length
)

print("Positions:", positions.numpy())

position_vectors = position_embedding(positions)

print("Position vector shape:", position_vectors.shape)

Positions: [0 1 2]
Position vector shape: (3, 16)


## 14. Combine token + Positional embeddings

In [14]:
position_vectors = tf.expand_dims(
    position_vectors,
    axis=0
)

print("Position vectors after expand:", position_vectors.shape)

combined_embedding = embedded + position_vectors

print("Combined embedding shape:", combined_embedding.shape)

Position vectors after expand: (1, 3, 16)
Combined embedding shape: (1, 3, 16)


## 15. Multi-Head Self-Attention + Causal Mask

In [15]:
attention_output = attention_layer(
    query=combined_embedding,
    key=combined_embedding,
    value=combined_embedding,
    use_causal_mask=True
)

print("Attention output shape:", attention_output.shape)

Attention output shape: (1, 3, 16)


## 16. First Residual + LayerNorm

In [16]:
x = combined_embedding + attention_output

print("After residual shape:", x.shape)

x = layer_norm1(x)

print("After LayerNorm shape:", x.shape)

After residual shape: (1, 3, 16)
After LayerNorm shape: (1, 3, 16)


## 17. FFN

In [17]:
ffn_output = ffn(x)

print("FFN output shape:", ffn_output.shape)

FFN output shape: (1, 3, 16)


## 18. Second Residual + LayerNorm

In [18]:
x = x + ffn_output

x = layer_norm2(x)

print("Final transformer block output:", x.shape)

Final transformer block output: (1, 3, 16)


## 19. Produce logits

In [19]:
logits = output_layer(x)

print("Logits shape:", logits.shape)

Logits shape: (1, 3, 8)


## 20. Convert logits to probabilities

In [20]:
probabilities = tf.nn.softmax(logits, axis=-1)

print("Probabilities shape:", probabilities.shape)
print(probabilities.numpy())

Probabilities shape: (1, 3, 8)
[[[0.15557042 0.05753531 0.0809214  0.04295829 0.06216811 0.05764675
   0.18986298 0.35333672]
  [0.02293344 0.15730846 0.04694048 0.321708   0.08315878 0.25032112
   0.08666122 0.03096849]
  [0.03660285 0.34546736 0.05642173 0.01481261 0.3350274  0.0359827
   0.04575741 0.12992793]]]


## 21. Get predicted token IDs

In [21]:
predicted_token_ids = tf.argmax(
    probabilities,
    axis=-1
)

print("Predicted token IDs:", predicted_token_ids.numpy())

Predicted token IDs: [[7 3 1]]


## 22. Cross-Entropy Loss

In [22]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)
flat_targets = tf.reshape(targets, [-1])
flat_logits = tf.reshape(logits, [-1, vocab_size])

print("Flat targets shape:", flat_targets.shape)
print("Flat logits shape:", flat_logits.shape)

loss = loss_fn(
    flat_targets,
    flat_logits
)

print("Loss:", loss.numpy())

Flat targets shape: (3,)
Flat logits shape: (3, 8)
Loss: 2.9864151


## 23. Optimizer

In [23]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.01
)

print("Optimizer created.")

Optimizer created.


## 24. Train one step

In [24]:
with tf.GradientTape() as tape:

    # Token embedding
    embedded = token_embedding(inputs)

    # Positional embedding
    sequence_length = tf.shape(inputs)[1]
    positions = tf.range(0, sequence_length)

    position_vectors = position_embedding(positions)
    position_vectors = tf.expand_dims(position_vectors, axis=0)

    combined_embedding = embedded + position_vectors

    # Attention
    attention_output = attention_layer(
        query=combined_embedding,
        key=combined_embedding,
        value=combined_embedding,
        use_causal_mask=True
    )

    # Residual + LayerNorm
    x = combined_embedding + attention_output
    x = layer_norm1(x)

    # FFN
    ffn_output = ffn(x)

    # Residual + LayerNorm
    x = x + ffn_output
    x = layer_norm2(x)

    # Output
    logits = output_layer(x)

    # Loss
    flat_targets = tf.reshape(targets, [-1])
    flat_logits = tf.reshape(logits, [-1, vocab_size])

    loss = loss_fn(
        flat_targets,
        flat_logits
    )

print("Loss before update:", loss.numpy())

Loss before update: 2.9864151


## 25. Calculate gradients and update weights

In [25]:
trainable_variables = (
    token_embedding.trainable_variables
    + position_embedding.trainable_variables
    + attention_layer.trainable_variables
    + layer_norm1.trainable_variables
    + ffn.trainable_variables
    + layer_norm2.trainable_variables
    + output_layer.trainable_variables
)

gradients = tape.gradient(
    loss,
    trainable_variables
)

optimizer.apply_gradients(
    zip(gradients, trainable_variables)
)

print("Weights updated.")

Weights updated.


## 26. Train repeatedly

In [26]:
epochs = 500

for epoch in range(epochs):

    with tf.GradientTape() as tape:

        embedded = token_embedding(inputs)

        sequence_length = tf.shape(inputs)[1]
        positions = tf.range(0, sequence_length)

        position_vectors = position_embedding(positions)
        position_vectors = tf.expand_dims(
            position_vectors,
            axis=0
        )

        combined_embedding = embedded + position_vectors

        attention_output = attention_layer(
            query=combined_embedding,
            key=combined_embedding,
            value=combined_embedding,
            use_causal_mask=True
        )

        x = combined_embedding + attention_output
        x = layer_norm1(x)

        ffn_output = ffn(x)

        x = x + ffn_output
        x = layer_norm2(x)

        logits = output_layer(x)

        flat_targets = tf.reshape(targets, [-1])
        flat_logits = tf.reshape(
            logits,
            [-1, vocab_size]
        )

        loss = loss_fn(
            flat_targets,
            flat_logits
        )

    gradients = tape.gradient(
        loss,
        trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, trainable_variables)
    )

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1}, Loss: {loss.numpy():.4f}"
        )

Epoch 50, Loss: 0.0030
Epoch 100, Loss: 0.0015
Epoch 150, Loss: 0.0009
Epoch 200, Loss: 0.0006
Epoch 250, Loss: 0.0005
Epoch 300, Loss: 0.0004
Epoch 350, Loss: 0.0003
Epoch 400, Loss: 0.0002
Epoch 450, Loss: 0.0002
Epoch 500, Loss: 0.0002


## 27. Check predictions after training

In [27]:
probabilities = tf.nn.softmax(
    logits,
    axis=-1
)

predicted_token_ids = tf.argmax(
    probabilities,
    axis=-1
)

print("Targets:           ", targets.numpy())
print("Predicted token IDs:", predicted_token_ids.numpy())

Targets:            [[3 4 5]]
Predicted token IDs: [[3 4 5]]


## Step 1 — Create our tiny real-text dataset

In [28]:
text = """
I love machine learning.
Machine learning is interesting.
I love learning artificial intelligence.
Machine learning helps us build intelligent systems.
"""

print(text)


I love machine learning.
Machine learning is interesting.
I love learning artificial intelligence.
Machine learning helps us build intelligent systems.



## Step 2 — Basic tokenization

In [29]:
import re

tokens = re.findall(r"\b\w+\b", text.lower())

print("First 30 tokens:")
print(tokens[:30])

First 30 tokens:
['i', 'love', 'machine', 'learning', 'machine', 'learning', 'is', 'interesting', 'i', 'love', 'learning', 'artificial', 'intelligence', 'machine', 'learning', 'helps', 'us', 'build', 'intelligent', 'systems']


## Step 3 — Create vocabulary and token embedding

In [30]:
# Create token embedding using the REAL vocabulary size

vocabulary = sorted(set(tokens))
vocab_size = len(vocabulary)

token_embedding = layers.Embedding(
    input_dim=vocab_size,
    output_dim=embed_dim
)

# Output layer must also use the real vocabulary size
output_layer = layers.Dense(vocab_size)

print("Vocabulary size:", vocab_size)
print("Token embedding:", token_embedding)
print("Output layer units:", vocab_size)

Vocabulary size: 13
Token embedding: <Embedding name=embedding_2, built=False>
Output layer units: 13


## Step 4 — Create token → ID mapping

In [31]:
token_to_id = {
    token: idx
    for idx, token in enumerate(vocabulary)
}

print(token_to_id)

{'artificial': 0, 'build': 1, 'helps': 2, 'i': 3, 'intelligence': 4, 'intelligent': 5, 'interesting': 6, 'is': 7, 'learning': 8, 'love': 9, 'machine': 10, 'systems': 11, 'us': 12}


## Step 5 — Create ID → token mapping

In [32]:
id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}

print(id_to_token)

{0: 'artificial', 1: 'build', 2: 'helps', 3: 'i', 4: 'intelligence', 5: 'intelligent', 6: 'interesting', 7: 'is', 8: 'learning', 9: 'love', 10: 'machine', 11: 'systems', 12: 'us'}


## Step 6 — Convert our text into token IDs

In [33]:
token_ids = [
    token_to_id[token]
    for token in tokens
]

print("First 30 token IDs:")
print(token_ids[:30])

First 30 token IDs:
[3, 9, 10, 8, 10, 8, 7, 6, 3, 9, 8, 0, 4, 10, 8, 2, 12, 1, 5, 11]


## Step 7 — Sliding window

In [34]:
sequence_length = 5

input_sequences = []
target_sequences = []

for i in range(len(token_ids) - sequence_length):
    input_sequences.append(
        token_ids[i:i + sequence_length]
    )
    
    target_sequences.append(
        token_ids[i + 1:i + sequence_length + 1]
    )

print("First 3 input sequences:")
print(input_sequences[:3])

print("\nFirst 3 target sequences:")
print(target_sequences[:3])

First 3 input sequences:
[[3, 9, 10, 8, 10], [9, 10, 8, 10, 8], [10, 8, 10, 8, 7]]

First 3 target sequences:
[[9, 10, 8, 10, 8], [10, 8, 10, 8, 7], [8, 10, 8, 7, 6]]


## Step 8 — Convert to TensorFlow tensors

In [35]:
inputs = tf.constant(input_sequences, dtype=tf.int32)
targets = tf.constant(target_sequences, dtype=tf.int32)

print("Inputs shape:", inputs.shape)
print("Targets shape:", targets.shape)

Inputs shape: (15, 5)
Targets shape: (15, 5)


## Step 9 — Update positional embedding handling

In [36]:
sequence_length = tf.shape(inputs)[1]

In [37]:
embedded = token_embedding(inputs)

sequence_length = tf.shape(inputs)[1]

positions = tf.range(
    start=0,
    limit=sequence_length
)

position_vectors = position_embedding(positions)
position_vectors = tf.expand_dims(
    position_vectors,
    axis=0
)

combined_embedding = embedded + position_vectors

print("Embedded shape:", embedded.shape)
print("Position shape:", position_vectors.shape)
print("Combined shape:", combined_embedding.shape)

Embedded shape: (15, 5, 16)
Position shape: (1, 5, 16)
Combined shape: (15, 5, 16)


## Step 10 — Multi-Head Self-Attention

In [38]:
attention_output = attention_layer(
    query=combined_embedding,
    key=combined_embedding,
    value=combined_embedding,
    use_causal_mask=True
)

print("Attention output shape:", attention_output.shape)

Attention output shape: (15, 5, 16)


## Step 11 — Residual Connection + Layer Normalization

In [39]:
x = combined_embedding + attention_output

x = layer_norm1(x)

print("After Attention + Residual + LayerNorm:", x.shape)

After Attention + Residual + LayerNorm: (15, 5, 16)


## Step 12 — Feed forward network

In [40]:
ffn_output = ffn(x)

print("FFN output shape:", ffn_output.shape)

FFN output shape: (15, 5, 16)


## Step 13 — Second Residual + LayerNorm

In [41]:
x = x + ffn_output

x = layer_norm2(x)

print("After FFN + Residual + LayerNorm:", x.shape)

After FFN + Residual + LayerNorm: (15, 5, 16)


## Step 14 — Output Layer → Logits

In [42]:
logits = output_layer(x)

print("Logits shape:", logits.shape)

Logits shape: (15, 5, 13)


## Step 15 — Compare predictions with targets

In [43]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

loss = loss_fn(
    targets,
    logits
)

print("Loss:", loss.numpy())

Loss: 3.1796548


## Step 16 — Create the optimizer

In [44]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

print("Optimizer created.")

Optimizer created.


## Step 17 — See what parameters the model has

In [45]:
trainable_variables = (
    token_embedding.trainable_variables
    + position_embedding.trainable_variables
    + attention_layer.trainable_variables
    + layer_norm1.trainable_variables
    + ffn.trainable_variables
    + layer_norm2.trainable_variables
    + output_layer.trainable_variables
)

print("Number of trainable variables:", len(trainable_variables))

for variable in trainable_variables:
    print(variable.name, variable.shape)

Number of trainable variables: 20
embeddings (13, 16)
embeddings (10, 16)
kernel (16, 2, 8)
bias (2, 8)
kernel (16, 2, 8)
bias (2, 8)
kernel (16, 2, 8)
bias (2, 8)
kernel (2, 8, 16)
bias (16,)
gamma (16,)
beta (16,)
kernel (16, 32)
bias (32,)
kernel (32, 16)
bias (16,)
gamma (16,)
beta (16,)
kernel (16, 13)
bias (13,)


## Step 18 — Calculate gradients

In [46]:
with tf.GradientTape() as tape:

    # Token embedding
    embedded = token_embedding(inputs)

    # Positional embedding
    sequence_length = tf.shape(inputs)[1]

    positions = tf.range(
        start=0,
        limit=sequence_length
    )

    position_vectors = position_embedding(positions)
    position_vectors = tf.expand_dims(
        position_vectors,
        axis=0
    )

    combined_embedding = embedded + position_vectors

    # Multi-head self-attention
    attention_output = attention_layer(
        query=combined_embedding,
        key=combined_embedding,
        value=combined_embedding,
        use_causal_mask=True
    )

    # Residual + LayerNorm
    x = combined_embedding + attention_output
    x = layer_norm1(x)

    # FFN
    ffn_output = ffn(x)

    # Residual + LayerNorm
    x = x + ffn_output
    x = layer_norm2(x)

    # Output layer
    logits = output_layer(x)

    # Loss
    loss = loss_fn(
        targets,
        logits
    )

print("Loss:", loss.numpy())

Loss: 3.1796548


## Step 19 — Calculate gradients

In [47]:
gradients = tape.gradient(
    loss,
    trainable_variables
)

print("Number of gradients:", len(gradients))

Number of gradients: 20


In [48]:
print("First variable:")
print(trainable_variables[0].name)
print(trainable_variables[0].shape)

print("\nFirst gradient:")
print(gradients[0].shape)

First variable:
embeddings
(13, 16)

First gradient:
(13, 16)


## Step 20 — Updates Weights

In [49]:
optimizer.apply_gradients(
    zip(gradients, trainable_variables)
)

print("Weights updated!")

Weights updated!


## Step 21 — Create train step

In [50]:
def train_step(inputs, targets):

    with tf.GradientTape() as tape:

        # -------------------------
        # 1. Token Embedding
        # -------------------------
        embedded = token_embedding(inputs)

        # -------------------------
        # 2. Positional Embedding
        # -------------------------
        sequence_length = tf.shape(inputs)[1]

        positions = tf.range(
            start=0,
            limit=sequence_length
        )

        position_vectors = position_embedding(positions)
        position_vectors = tf.expand_dims(
            position_vectors,
            axis=0
        )

        combined_embedding = embedded + position_vectors

        # -------------------------
        # 3. Multi-Head Attention
        # -------------------------
        attention_output = attention_layer(
            query=combined_embedding,
            key=combined_embedding,
            value=combined_embedding,
            use_causal_mask=True
        )

        # -------------------------
        # 4. Residual + LayerNorm
        # -------------------------
        x = combined_embedding + attention_output
        x = layer_norm1(x)

        # -------------------------
        # 5. Feed Forward Network
        # -------------------------
        ffn_output = ffn(x)

        # -------------------------
        # 6. Residual + LayerNorm
        # -------------------------
        x = x + ffn_output
        x = layer_norm2(x)

        # -------------------------
        # 7. Output → Logits
        # -------------------------
        logits = output_layer(x)

        # -------------------------
        # 8. Loss
        # -------------------------
        loss = loss_fn(
            targets,
            logits
        )

    # -------------------------
    # 9. Gradients
    # -------------------------
    gradients = tape.gradient(
        loss,
        trainable_variables
    )

    # -------------------------
    # 10. Update weights
    # -------------------------
    optimizer.apply_gradients(
        zip(gradients, trainable_variables)
    )

    return loss

## Step 22 — Test one training step

In [51]:
loss = train_step(
    inputs,
    targets
)

print("Loss after one training step:", loss.numpy())

Loss after one training step: 3.0456886


## Step 23 — Train for multiple epochs

In [52]:
epochs = 500

for epoch in range(epochs):

    loss = train_step(
        inputs,
        targets
    )

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs} - Loss: {loss.numpy():.4f}"
        )

Epoch 50/500 - Loss: 0.6914
Epoch 100/500 - Loss: 0.3275
Epoch 150/500 - Loss: 0.2303
Epoch 200/500 - Loss: 0.1985
Epoch 250/500 - Loss: 0.1847
Epoch 300/500 - Loss: 0.1769
Epoch 350/500 - Loss: 0.1720
Epoch 400/500 - Loss: 0.1686
Epoch 450/500 - Loss: 0.1662
Epoch 500/500 - Loss: 0.1644


## Step 24 — Create a helper function to convert text → IDs

In [53]:
def encode_text(text):
    tokens = re.findall(r"\b\w+\b", text.lower())

    return [
        token_to_id[token]
        for token in tokens
    ]

In [54]:
prompt_ids = encode_text("I love")

print("Prompt token IDs:", prompt_ids)

Prompt token IDs: [3, 9]


## Step 24.2 — Create ID → text function

In [55]:
def decode_tokens(token_ids):
    return " ".join(
        id_to_token[int(token_id)]
        for token_id in token_ids
    )

In [56]:
print(decode_tokens(prompt_ids))

i love


## Step 24.3 — Predict the next token

In [57]:
def predict_next_token(input_ids):

    input_tensor = tf.constant(
        [input_ids],
        dtype=tf.int32
    )

    # Token embedding
    embedded = token_embedding(input_tensor)

    # Position embedding
    sequence_length = tf.shape(input_tensor)[1]

    positions = tf.range(
        start=0,
        limit=sequence_length
    )

    position_vectors = position_embedding(positions)
    position_vectors = tf.expand_dims(
        position_vectors,
        axis=0
    )

    combined_embedding = embedded + position_vectors

    # Attention
    attention_output = attention_layer(
        query=combined_embedding,
        key=combined_embedding,
        value=combined_embedding,
        use_causal_mask=True
    )

    # Residual + LayerNorm
    x = combined_embedding + attention_output
    x = layer_norm1(x)

    # FFN
    ffn_output = ffn(x)

    # Residual + LayerNorm
    x = x + ffn_output
    x = layer_norm2(x)

    # Output logits
    logits = output_layer(x)

    # Get only the LAST position
    next_token_logits = logits[:, -1, :]

    # Convert to probabilities
    probabilities = tf.nn.softmax(
        next_token_logits,
        axis=-1
    )

    # Select highest probability
    next_token_id = tf.argmax(
        probabilities,
        axis=-1
    )

    return int(next_token_id[0]), probabilities

## Step 24.4 — Test it

In [58]:
prompt_ids = encode_text("I love")

next_token_id, probabilities = predict_next_token(
    prompt_ids
)

print("Prompt:", decode_tokens(prompt_ids))
print("Predicted token:", id_to_token[next_token_id])
print("Predicted token ID:", next_token_id)

Prompt: i love
Predicted token: learning
Predicted token ID: 8


## Step 25 — Generate multiple token

In [59]:
def generate_text(prompt, max_new_tokens=5):

    input_ids = encode_text(prompt)

    for _ in range(max_new_tokens):

        next_token_id, probabilities = predict_next_token(
            input_ids
        )

        input_ids.append(next_token_id)

    return decode_tokens(input_ids)

In [60]:
generated_text = generate_text(
    "I love",
    max_new_tokens=5
)

print("Generated text:")
print(generated_text)

Generated text:
i love learning artificial intelligence machine learning


## Step 26.1 — Modify our prediction function: Add temperature

In [61]:
def predict_next_token_temperature(input_ids, temperature=1.0):

    input_tensor = tf.constant(
        [input_ids],
        dtype=tf.int32
    )

    # Token embedding
    embedded = token_embedding(input_tensor)

    # Positional embedding
    sequence_length = tf.shape(input_tensor)[1]

    positions = tf.range(
        start=0,
        limit=sequence_length
    )

    position_vectors = position_embedding(positions)
    position_vectors = tf.expand_dims(
        position_vectors,
        axis=0
    )

    combined_embedding = embedded + position_vectors

    # Self-attention
    attention_output = attention_layer(
        query=combined_embedding,
        key=combined_embedding,
        value=combined_embedding,
        use_causal_mask=True
    )

    # Residual + LayerNorm
    x = combined_embedding + attention_output
    x = layer_norm1(x)

    # FFN
    ffn_output = ffn(x)

    # Residual + LayerNorm
    x = x + ffn_output
    x = layer_norm2(x)

    # Output logits
    logits = output_layer(x)

    # Last token position
    next_token_logits = logits[:, -1, :]

    # Apply temperature
    scaled_logits = next_token_logits / temperature

    # Convert to probabilities
    probabilities = tf.nn.softmax(
        scaled_logits,
        axis=-1
    )

    # Sample from probability distribution
    next_token_id = tf.random.categorical(
        scaled_logits,
        num_samples=1
    )

    return int(next_token_id[0, 0]), probabilities

In [62]:
prompt_ids = encode_text("I love")

for temperature in [0.5, 1.0, 1.5]:

    next_token_id, probabilities = predict_next_token_temperature(
        prompt_ids,
        temperature=temperature
    )

    print(
        "Temperature:",
        temperature,
        "→",
        id_to_token[next_token_id]
    )

Temperature: 0.5 → machine
Temperature: 1.0 → machine
Temperature: 1.5 → machine


## Step 27.1 — Create a Top-K prediction function

In [63]:
def predict_next_token_top_k(input_ids, temperature=1.0, top_k=3):

    input_tensor = tf.constant(
        [input_ids],
        dtype=tf.int32
    )

    # Token embedding
    embedded = token_embedding(input_tensor)

    # Positional embedding
    sequence_length = tf.shape(input_tensor)[1]

    positions = tf.range(
        start=0,
        limit=sequence_length
    )

    position_vectors = position_embedding(positions)
    position_vectors = tf.expand_dims(
        position_vectors,
        axis=0
    )

    combined_embedding = embedded + position_vectors

    # Self-attention
    attention_output = attention_layer(
        query=combined_embedding,
        key=combined_embedding,
        value=combined_embedding,
        use_causal_mask=True
    )

    # Residual + LayerNorm
    x = combined_embedding + attention_output
    x = layer_norm1(x)

    # FFN
    ffn_output = ffn(x)

    # Residual + LayerNorm
    x = x + ffn_output
    x = layer_norm2(x)

    # Output logits
    logits = output_layer(x)

    # Last position
    next_token_logits = logits[:, -1, :]

    # Temperature
    scaled_logits = next_token_logits / temperature

    # Top-K
    values, indices = tf.math.top_k(
        scaled_logits,
        k=top_k
    )

    # Sample only from Top-K
    sampled_index = tf.random.categorical(
        values,
        num_samples=1
    )

    sampled_index = sampled_index[0, 0]

    # Convert position inside Top-K back to vocabulary ID
    next_token_id = indices[0, sampled_index]

    return int(next_token_id), values

## Step 27.2 — Test it

In [64]:
prompt_ids = encode_text("I love")

next_token_id, top_k_values = predict_next_token_top_k(
    prompt_ids,
    temperature=1.0,
    top_k=3
)

print("Prompt:", decode_tokens(prompt_ids))
print("Predicted token:", id_to_token[next_token_id])

Prompt: i love
Predicted token: machine


## Step 28.1 — Create Top-P function

In [65]:
def predict_next_token_top_p(
    input_ids,
    temperature=1.0,
    top_p=0.9
):

    input_tensor = tf.constant(
        [input_ids],
        dtype=tf.int32
    )

    # -------------------------
    # 1. Token Embedding
    # -------------------------
    embedded = token_embedding(input_tensor)

    # -------------------------
    # 2. Positional Embedding
    # -------------------------
    sequence_length = tf.shape(input_tensor)[1]

    positions = tf.range(
        start=0,
        limit=sequence_length
    )

    position_vectors = position_embedding(positions)
    position_vectors = tf.expand_dims(
        position_vectors,
        axis=0
    )

    combined_embedding = embedded + position_vectors

    # -------------------------
    # 3. Self-Attention
    # -------------------------
    attention_output = attention_layer(
        query=combined_embedding,
        key=combined_embedding,
        value=combined_embedding,
        use_causal_mask=True
    )

    # -------------------------
    # 4. Residual + LayerNorm
    # -------------------------
    x = combined_embedding + attention_output
    x = layer_norm1(x)

    # -------------------------
    # 5. FFN
    # -------------------------
    ffn_output = ffn(x)

    # -------------------------
    # 6. Residual + LayerNorm
    # -------------------------
    x = x + ffn_output
    x = layer_norm2(x)

    # -------------------------
    # 7. Output Layer
    # -------------------------
    logits = output_layer(x)

    # Last position
    next_token_logits = logits[:, -1, :]

    # -------------------------
    # 8. Temperature
    # -------------------------
    scaled_logits = next_token_logits / temperature

    # Remove batch dimension
    scaled_logits = scaled_logits[0]

    # -------------------------
    # 9. Convert to probabilities
    # -------------------------
    probabilities = tf.nn.softmax(
        scaled_logits
    )

    # -------------------------
    # 10. Sort probabilities
    # -------------------------
    sorted_indices = tf.argsort(
        probabilities,
        direction="DESCENDING"
    )

    sorted_probabilities = tf.gather(
        probabilities,
        sorted_indices
    )

    # -------------------------
    # 11. Cumulative probability
    # -------------------------
    cumulative_probabilities = tf.cumsum(
        sorted_probabilities
    )

    # -------------------------
    # 12. Create Top-P mask
    # -------------------------
    mask = cumulative_probabilities <= top_p

    # Always keep the highest-probability token
    mask = tf.concat(
        [
            tf.constant([True]),
            mask[1:]
        ],
        axis=0
    )

    # -------------------------
    # 13. Filter probabilities
    # -------------------------
    filtered_probabilities = tf.where(
        mask,
        sorted_probabilities,
        tf.zeros_like(sorted_probabilities)
    )

    # -------------------------
    # 14. Renormalize
    # -------------------------
    filtered_probabilities = (
        filtered_probabilities /
        tf.reduce_sum(filtered_probabilities)
    )

    # -------------------------
    # 15. Sample
    # -------------------------
    sampled_position = tf.random.categorical(
        tf.math.log(
            tf.expand_dims(
                filtered_probabilities,
                axis=0
            )
        ),
        num_samples=1
    )

    sampled_position = sampled_position[0, 0]

    # -------------------------
    # 16. Convert position
    #     back to token ID
    # -------------------------
    next_token_id = sorted_indices[sampled_position]

    return int(next_token_id)

## Step 28.2 — Test it

In [66]:
prompt_ids = encode_text("I love")

next_token_id = predict_next_token_top_p(
    prompt_ids,
    temperature=1.0,
    top_p=0.9
)

print("Prompt:", decode_tokens(prompt_ids))
print("Predicted token:", id_to_token[next_token_id])

Prompt: i love
Predicted token: learning


## Auto regressive text generation

In [67]:
# Starting prompt
generated_tokens = [2]   # "I"

print("Starting tokens:", generated_tokens)

Starting tokens: [2]


In [72]:
def generate_text(
    input_ids,
    max_new_tokens=5,
    temperature=1.0,
    top_p=0.9
):
    
    generated_ids = input_ids.copy()

    for _ in range(max_new_tokens):

        # Predict one next token
        next_token_id = predict_next_token_top_p(
            generated_ids,
            temperature=temperature,
            top_p=top_p
        )

        # Add predicted token to sequence
        generated_ids.append(next_token_id)

        # Stop if END token is generated
        if next_token_id == 7:
            break

    return generated_ids

In [71]:
id_to_token = {
    0: "<PAD>",
    1: "<START>",
    2: "I",
    3: "love",
    4: "AI",
    5: "and",
    6: "ML",
    7: "<END>"
}

generated_ids = generate_text(
    [2],
    max_new_tokens=4,
    temperature=1.0,
    top_p=0.9
)

generated_text = [
    id_to_token[token_id]
    for token_id in generated_ids
]

print("Generated IDs:", generated_ids)
print("Generated text:", generated_text)

KeyError: 12

In [73]:
print("Vocabulary size in id_to_token:", len(id_to_token))
print("Model output vocabulary size:", output_layer.units)
print("Generated IDs:", generated_ids)

Vocabulary size in id_to_token: 8
Model output vocabulary size: 13
Generated IDs: [2, 12, 1, 5, 11]


In [74]:
print("vocab_size:", vocab_size)

print("token_to_id:")
print(token_to_id)

vocab_size: 13
token_to_id:
{'artificial': 0, 'build': 1, 'helps': 2, 'i': 3, 'intelligence': 4, 'intelligent': 5, 'interesting': 6, 'is': 7, 'learning': 8, 'love': 9, 'machine': 10, 'systems': 11, 'us': 12}


In [75]:
print("output layer units:", output_layer.units)

output layer units: 13


In [76]:
id_to_token = {
    token_id: token
    for token, token_id in token_to_id.items()
}

print(id_to_token)

{0: 'artificial', 1: 'build', 2: 'helps', 3: 'i', 4: 'intelligence', 5: 'intelligent', 6: 'interesting', 7: 'is', 8: 'learning', 9: 'love', 10: 'machine', 11: 'systems', 12: 'us'}


In [77]:
generated_text = [
    id_to_token[token_id]
    for token_id in generated_ids
]

print("Generated IDs:", generated_ids)
print("Generated text:", " ".join(generated_text))

Generated IDs: [2, 12, 1, 5, 11]
Generated text: helps us build intelligent systems


In [78]:
for temperature in [0.2, 0.7, 1.0, 1.5]:

    generated_ids = generate_text(
        [2],                  # "helps"
        max_new_tokens=4,
        temperature=temperature,
        top_p=0.9
    )

    generated_text = [
        id_to_token[token_id]
        for token_id in generated_ids
    ]

    print(
        f"Temperature {temperature}:",
        " ".join(generated_text)
    )

Temperature 0.2: helps us build intelligent systems
Temperature 0.7: helps us build intelligent systems
Temperature 1.0: helps us build intelligent systems
Temperature 1.5: helps us build us build


In [79]:
input_ids = [2, 12, 1]  # helps us build

# Get probabilities by calling the model logic
input_tensor = tf.constant([input_ids], dtype=tf.int32)

embedded = token_embedding(input_tensor)

sequence_length = tf.shape(input_tensor)[1]

positions = tf.range(
    start=0,
    limit=sequence_length
)

position_vectors = position_embedding(positions)
position_vectors = tf.expand_dims(position_vectors, axis=0)

combined_embedding = embedded + position_vectors

attention_output = attention_layer(
    query=combined_embedding,
    key=combined_embedding,
    value=combined_embedding,
    use_causal_mask=True
)

x = combined_embedding + attention_output
x = layer_norm1(x)

ffn_output = ffn(x)

x = x + ffn_output
x = layer_norm2(x)

logits = output_layer(x)

# Only the final position
next_token_logits = logits[:, -1, :]

probabilities = tf.nn.softmax(
    next_token_logits[0]
)

for token_id, probability in enumerate(probabilities.numpy()):
    print(
        token_id,
        id_to_token[token_id],
        f"{probability:.4f}"
    )

0 artificial 0.0023
1 build 0.0016
2 helps 0.0004
3 i 0.0030
4 intelligence 0.0037
5 intelligent 0.9650
6 interesting 0.0014
7 is 0.0011
8 learning 0.0025
9 love 0.0000
10 machine 0.0003
11 systems 0.0059
12 us 0.0127


In [80]:
input_ids = [2, 12, 1]  # helps us build

# Get the logits from the model
input_tensor = tf.constant([input_ids], dtype=tf.int32)

embedded = token_embedding(input_tensor)

sequence_length = tf.shape(input_tensor)[1]

positions = tf.range(
    start=0,
    limit=sequence_length
)

position_vectors = position_embedding(positions)
position_vectors = tf.expand_dims(position_vectors, axis=0)

combined_embedding = embedded + position_vectors

attention_output = attention_layer(
    query=combined_embedding,
    key=combined_embedding,
    value=combined_embedding,
    use_causal_mask=True
)

x = combined_embedding + attention_output
x = layer_norm1(x)

ffn_output = ffn(x)

x = x + ffn_output
x = layer_norm2(x)

logits = output_layer(x)

# Only the last position
next_token_logits = logits[:, -1, :][0]

for temperature in [0.2, 0.7, 1.0, 1.5]:

    scaled_logits = next_token_logits / temperature

    probabilities = tf.nn.softmax(scaled_logits)

    print(f"\nTemperature: {temperature}")

    for token_id, probability in enumerate(probabilities.numpy()):
        print(
            f"{id_to_token[token_id]:12s} "
            f"{probability:.4f}"
        )


Temperature: 0.2
artificial   0.0000
build        0.0000
helps        0.0000
i            0.0000
intelligence 0.0000
intelligent  1.0000
interesting  0.0000
is           0.0000
learning     0.0000
love         0.0000
machine      0.0000
systems      0.0000
us           0.0000

Temperature: 0.7
artificial   0.0002
build        0.0001
helps        0.0000
i            0.0003
intelligence 0.0004
intelligent  0.9960
interesting  0.0001
is           0.0001
learning     0.0002
love         0.0000
machine      0.0000
systems      0.0007
us           0.0020

Temperature: 1.0
artificial   0.0023
build        0.0016
helps        0.0004
i            0.0030
intelligence 0.0037
intelligent  0.9650
interesting  0.0014
is           0.0011
learning     0.0025
love         0.0000
machine      0.0003
systems      0.0059
us           0.0127

Temperature: 1.5
artificial   0.0147
build        0.0113
helps        0.0048
i            0.0176
intelligence 0.0202
intelligent  0.8194
interesting  0.0107
is      

## Top-P

In [81]:
temperature = 1.5
top_p = 0.9

scaled_logits = next_token_logits / temperature
probabilities = tf.nn.softmax(scaled_logits)

# Sort probabilities
sorted_indices = tf.argsort(
    probabilities,
    direction="DESCENDING"
)

sorted_probabilities = tf.gather(
    probabilities,
    sorted_indices
)

# Cumulative probability
cumulative_probabilities = tf.cumsum(
    sorted_probabilities
)

print("Token        Probability    Cumulative")

for i in range(len(sorted_indices)):
    token_id = int(sorted_indices[i])
    probability = float(sorted_probabilities[i])
    cumulative = float(cumulative_probabilities[i])

    print(
        f"{id_to_token[token_id]:12s} "
        f"{probability:.4f}        "
        f"{cumulative:.4f}"
    )

Token        Probability    Cumulative
intelligent  0.8194        0.8194
us           0.0456        0.8650
systems      0.0275        0.8925
intelligence 0.0202        0.9127
i            0.0176        0.9303
learning     0.0153        0.9456
artificial   0.0147        0.9603
build        0.0113        0.9715
interesting  0.0107        0.9823
is           0.0091        0.9913
helps        0.0048        0.9961
machine      0.0033        0.9995
love         0.0005        1.0000
